<a href="https://colab.research.google.com/github/dcthyun0308/ESAA/blob/main/ESAA_OB_week04_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 저해상도 조류 이미지 분류 AI 경진대회

1. 코드 흐름 요약
- 전처리 및 EDA :결측치 확인하고, 범주형 변수와 수치형 변수 분리
- 피처 엔지니어링 : 기존 칼럼들을 조합하여 산사태에 영향을 줄 수 있는 새로운 지표를 생성.
- 모델링
- 교차 검증
- 앙상블

2. 새롭게 알게 된 점 및 배울 점
- 지형 데이터의 특성 활용: 단순히 수치로만 접근하는 것이 아니라, 지형학적으로 산사태에 중요한 변수를 우선순위에 두고 피처를 생성하는 점이 인상적이었음.
- StratifiedKFold의 중요성: 산사태 발생 여부(Target)의 클래스 불균형이 있을 수 있는 상황에서, 타겟 비율을 유지하며 학습시키는 교차 검증의 필수성을 다시 한번 확인함.

3. 어려운 내용
- 데이터 내에 존재하는 수많은 지질학적 용어와 변수들 사이의 관계를 파악하여 유의미한 피처를 뽑아내는 과정이 가장 난이도가 높게 느껴짐.

4. 주요 코드
# StratifiedKFold를 활용한 모델 학습 및 검증 로직 예시
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(train_x.shape[0])
test_preds = np.zeros(test_x.shape[0])

for fold, (train_idx, val_idx) in enumerate(skf.split(train_x, train_y)):
    X_train, X_val = train_x.iloc[train_idx], train_x.iloc[val_idx]
    y_train, y_val = train_y.iloc[train_idx], train_y.iloc[val_idx]
    
    # 모델 정의 및 학습 (예: CatBoost)
    model = CatBoostClassifier(iterations=1000, silent=True, random_state=42)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=100)
    
    # 검증 데이터와 테스트 데이터에 대한 예측값 누적
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(test_x)[:, 1] / 5 # Fold 수만큼 나누어 평균
